In [1]:
import pandas as pd
import numpy as np
import random
import time
import itertools

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

In [2]:
np.random.seed(19)
pio.renderers.default = 'iframe'

In [3]:
order_details = pd.read_csv('data/order_details.csv', encoding='windows-1251')
products = pd.read_csv('data/products.csv', encoding='windows-1251')
categories = pd.read_csv('data/categories.csv', encoding='windows-1251')

In [4]:
order_details

,orderID,productID,unitPrice,quantity,discount
0,10248,11,14.00,12,0.00
1,10248,42,9.80,10,0.00
2,10248,72,34.80,5,0.00
3,10249,14,18.60,9,0.00
4,10249,51,42.40,40,0.00
...,...,...,...,...,...
2150,11077,64,33.25,2,0.03
2151,11077,66,17.00,1,0.00
2152,11077,73,15.00,2,0.01
2153,11077,75,7.75,4,0.00


In [5]:
data = order_details.merge(products, how='inner', on='productID')
df = data.merge(categories, how='inner', on='categoryID')
df = df[['productName', 'categoryName', 'unitPrice_x', 'quantity']]
df

,productName,categoryName,unitPrice_x,quantity
0,Queso Cabrales,Dairy Products,14.00,12
1,Singaporean Hokkien Fried Mee,Grains & Cereals,9.80,10
2,Mozzarella di Giovanni,Dairy Products,34.80,5
3,Tofu,Produce,18.60,9
4,Manjimup Dried Apples,Produce,42.40,40
...,...,...,...,...
2150,Wimmers gute Semmelknцdel,Grains & Cereals,33.25,2
2151,Louisiana Hot Spiced Okra,Condiments,17.00,1
2152,Rцd Kaviar,Seafood,15.00,2
2153,Rhцnbrдu Klosterbier,Beverages,7.75,4


In [6]:
df_clean = df[(df['unitPrice_x'] > 0) & (df['quantity'] > 0)].copy()

def remove_price_outliers(group):
    q1 = group['unitPrice_x'].quantile(0.25)
    q3 = group['unitPrice_x'].quantile(0.75)
    iqr = q3 - q1
    if iqr == 0:
        return group
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return group[(group['unitPrice_x'] >= lower) & (group['unitPrice_x'] <= upper)]

cleaned_groups = [
    remove_price_outliers(sub)
    for _, sub in df_clean.groupby('productName', sort=False)
]
df_clean = pd.concat(cleaned_groups, ignore_index=True)

assert 'productName' in df_clean.columns, \
    f"productName пропала. Колонки: {df_clean.columns.tolist()}"

In [7]:
product_stats = df_clean.groupby('productName').agg(
    n_obs=('unitPrice_x', 'count'),
    price_mean=('unitPrice_x', 'mean'),
    price_std=('unitPrice_x', 'std'),
    price_min=('unitPrice_x', 'min'),
    price_max=('unitPrice_x', 'max'),
    qty_mean=('quantity', 'mean'),
    qty_sum=('quantity', 'sum'),
).reset_index()

product_stats

,productName,n_obs,price_mean,price_std,price_min,price_max,qty_mean,qty_sum
0,Alice Mutton,37,36.470270,3.701716,31.2,39.00,26.432432,978
1,Aniseed Syrup,9,10.000000,0.000000,10.0,10.00,25.333333,228
2,Boston Crab Meat,41,17.226829,1.743133,14.7,18.40,26.902439,1103
3,Camembert Pierrot,51,32.133333,3.064811,27.2,34.00,30.921569,1577
4,Carnarvon Tigers,27,59.722222,5.295741,50.0,62.50,19.962963,539
...,...,...,...,...,...,...,...,...
72,Uncle Bob's Organic Dried Pears,29,29.172414,2.105587,24.0,30.00,26.310345,763
73,Valkoinen suklaa,10,14.950000,1.678293,13.0,16.25,23.500000,235
74,Vegie-spread,17,40.794118,4.334811,35.1,43.90,26.176471,445
75,Wimmers gute Semmelknцdel,30,31.033333,3.188431,26.6,33.25,24.666667,740


In [8]:
MIN_OBS = 10
valid_products = product_stats[
    (product_stats['n_obs'] >= MIN_OBS) &
    (product_stats['price_std'] > 0)
]['productName'].tolist()

print(f"Товаров после очистки и фильтрации: {len(valid_products)}")
print(product_stats[product_stats['productName'].isin(valid_products)]
      [['productName', 'n_obs', 'price_mean', 'price_std', 'qty_mean']]
      .to_string(index=False))

df_selected = df_clean[df_clean['productName'].isin(valid_products)].reset_index(drop=True)

Товаров после очистки и фильтрации: 70
                     productName  n_obs  price_mean  price_std  qty_mean
                    Alice Mutton     37   36.470270   3.701716 26.432432
                Boston Crab Meat     41   17.226829   1.743133 26.902439
               Camembert Pierrot     51   32.133333   3.064811 30.921569
                Carnarvon Tigers     27   59.722222   5.295741 19.962963
                            Chai     38   17.147368   1.551065 21.789474
                           Chang     44   17.877273   1.753782 24.022727
                Chartreuse verte     30   16.680000   1.764477 26.433333
    Chef Anton's Cajun Seasoning     20   20.680000   2.068714 22.650000
          Chef Anton's Gumbo Mix     10   19.610000   2.246330 29.800000
                   Cфte de Blaye     24  245.933333  25.377338 25.958333
          Escargots de Bourgogne     18   12.661111   1.133650 29.666667
                        Filo Mix     29    6.758621   0.538196 17.241379
            

In [9]:
DEFAULT_ELASTICITY = -2.0
COST_MARGIN = 0.6
R2_THRESHOLD = 0.1

demand_params = {}

for product in valid_products:
    subdata = df_selected[df_selected['productName'] == product].copy()

    p_mean = float(subdata['unitPrice_x'].mean())
    q_mean = float(subdata['quantity'].mean())
    p_max = float(subdata['unitPrice_x'].max())
    cost = p_mean * COST_MARGIN

    X = subdata[['unitPrice_x']].values
    y = subdata['quantity'].values

    lr = LinearRegression()
    lr.fit(X, y)
    r2 = r2_score(y, lr.predict(X))

    a_reg = float(lr.intercept_)
    b_reg = float(lr.coef_[0])

    if r2 > R2_THRESHOLD and b_reg < 0:
        a = a_reg
        b = b_reg
        sourde = f'regression (R2={r2:.2f})'
    else:
        b = DEFAULT_ELASTICITY * (q_mean / p_mean)
        a = q_mean - b * p_mean
        source = f'elasticity ({DEFAULT_ELASTICITY})'

    demand_params[product] = {
        'a': a,
        'b': b,
        'cost': float(cost),
        'mean_price': p_mean,
        'max_price': p_max,
        'q_mean': q_mean,
        'source': source,
        'r2': float(r2),
    }

In [10]:
n_reg = sum(1 for m in demand_params.values() if m['source'].startswith('regression'))
n_el = len(demand_params) - n_reg
print(f"\nОткалибровано регрессией: {n_reg}, эластичностью: {n_el}")

for name, m in demand_params.items():
    assert m['b'] < 0, f"{name}: b={m['b']:.4f} >= 0"
    assert m['a'] > 0, f"{name}: a={m['a']:.2f} <= 0"
    assert m['max_price'] > m['cost'], f"{name}: max_price <= cost"


Откалибровано регрессией: 0, эластичностью: 70


## Алгоритм искусственной пчелинной колонии

In [11]:
models_df = pd.DataFrame([
    {
        'product': name,
        'a': m['a'],
        'b': m['b'],
        'cost': m['cost'],
        'mean_price': m['mean_price'],
        'max_price': m['max_price'],
        'q_mean': m['q_mean'],
        'source': m['source'],
        'r2': m['r2'],
    }
    for name, m in demand_params.items()
])
print("\nСводная таблица моделей спроса:")
models_df


Сводная таблица моделей спроса:


,product,a,b,cost,mean_price,max_price,q_mean,source,r2
0,Alice Mutton,79.297297,-1.449533,21.882162,36.470270,39.00,26.432432,elasticity (-2.0),0.004223
1,Boston Crab Meat,80.707317,-3.123319,10.336098,17.226829,18.40,26.902439,elasticity (-2.0),0.000945
2,Camembert Pierrot,92.764706,-1.924579,19.280000,32.133333,34.00,30.921569,elasticity (-2.0),0.012054
3,Carnarvon Tigers,59.888889,-0.668527,35.833333,59.722222,62.50,19.962963,elasticity (-2.0),0.014705
4,Chai,65.368421,-2.541436,10.288421,17.147368,18.00,21.789474,elasticity (-2.0),0.006205
...,...,...,...,...,...,...,...,...,...
65,Uncle Bob's Organic Dried Pears,78.931034,-1.803783,17.503448,29.172414,30.00,26.310345,elasticity (-2.0),0.025093
66,Valkoinen suklaa,70.500000,-3.143813,8.970000,14.950000,16.25,23.500000,elasticity (-2.0),0.000454
67,Vegie-spread,88.196453,-1.520317,24.476471,40.794118,43.90,26.176471,elasticity (-2.0),0.100588
68,Wimmers gute Semmelknцdel,74.000000,-1.589689,18.620000,31.033333,33.25,24.666667,elasticity (-2.0),0.000403


In [12]:
class ArtificialBeeColony():
    def __init__(self, models, num_bees:int=30, max_iter:int=100,limit:int=20, epsilon:float=10**-6):
        self.n_evaluations = 0
    
        self.models = models
        self.num_bees = num_bees
        self.max_iter = max_iter
        self.limit = limit
        self.epsilon = epsilon

        self.product_names = list(models.keys())
        self.num_products = len(self.product_names)

        for name in self.product_names:
            m = models[name]
            assert m['b'] < 0, f"{name}: b должно быть < 0 (убывающий спрос)"
            assert m['max_price'] > m['cost'], f"{name}: max_price должно быть > cost"
            assert m['a'] > 0, f"{name}: a должно быть > 0"
        
        self.lower_bounds = np.array([self.models[name]['cost'] for name in self.product_names], dtype=float)
        self.upper_bounds = np.array([self.models[name]['max_price'] * 1.2 for name in self.product_names], dtype=float)

        self.population = np.random.uniform(
            low=self.lower_bounds,
            high=self.upper_bounds,
            size=(self.num_bees, self.num_products)
        )
        self.profits = np.zeros(self.num_bees, dtype=float)
        self.fitness = np.zeros(self.num_bees, dtype=float)
        self.trials = np.zeros(self.num_bees, dtype=int)

        self._evaluate_population()
        
        best_idx = int(np.argmax(self.profits))
        self.best_profit = float(self.profits[best_idx])
        self.best_prices = np.copy(self.population[best_idx])

        self.history = []
        self.best_prices_history = []

    def _clip_solution(self, solution: np.ndarray):
        return np.clip(solution, self.lower_bounds, self.upper_bounds)

    def _calculate_profit(self, prices):
        self.n_evaluations += 1
        return self.calculate_profit(prices, self.models)
    
    def _evaluate_population(self):
        for i in range(self.num_bees):
            self.profits[i] = self._calculate_profit(self.population[i])
        self._update_fitness()


    def _update_fitness(self):
        min_profit = float(np.min(self.profits))
        self.fitness = self.profits - min_profit + self.epsilon

    
    def _try_improve_bee(self, bee_index: int):
        i = bee_index

        k = np.random.randint(0, self.num_bees)
        while k == i:
            k = np.random.randint(0, self.num_bees)

        j = np.random.randint(0, self.num_products)

        phi = np.random.uniform(-1.0, 1.0)

        candidate = np.copy(self.population[i])
        candidate[j] = self.population[i][j] + phi * (self.population[i][j] - self.population[k][j])
        candidate = self._clip_solution(candidate)
        candidate_profit = self._calculate_profit(candidate)

        if candidate_profit > self.profits[i]:
            self.population[i] = candidate
            self.profits[i] = candidate_profit
            self.trials[i] = 0
            return True

        self.trials[i] += 1
        return False

    
    def _employed_bees_phase(self):
        for i in range(self.num_bees):
            self._try_improve_bee(i)
            

    def _onlooker_bees_phase(self):
        self._update_fitness()
        fitness_sum = float(np.sum(self.fitness))

        if fitness_sum <= 0:
            probabilities = np.ones(self.num_bees) / self.num_bees
        else:
            probabilities = self.fitness / fitness_sum

        for _ in range(self.num_bees):
            i = int(np.random.choice(self.num_bees, p=probabilities))
            self._try_improve_bee(i)
            

    def _scout_bees_phase(self):
        for i in range(self.num_bees):
            if self.trials[i] > self.limit:
                self.population[i] = np.random.uniform(
                    low=self.lower_bounds,
                    high=self.upper_bounds
                )

                self.profits[i] = self._calculate_profit(self.population[i])
                self.trials[i] = 0


    def _update_best(self):
        current_best_index = int(np.argmax(self.profits))
        current_best_profit = self.profits[current_best_index]

        if current_best_profit > self.best_profit:            
            self.best_profit = float(current_best_profit)
            self.best_prices = np.copy(self.population[current_best_index])
        
        self.best_prices_history.append(self.best_prices)
        self.history.append(self.best_profit)

    
    def optimize(self):
        for _ in range(self.max_iter):
            self._update_fitness()
            self._employed_bees_phase()
            self._onlooker_bees_phase()
            self._scout_bees_phase()
            self._update_best()

        return self.best_prices, self.best_profit

    
    @staticmethod
    def calculate_profit(prices, models):
        total_profit = 0.0

        for i, product_name in enumerate(list(models.keys())):
            model = models[product_name]

            price = float(prices[i])
            a = model['a']
            b = model['b']
            cost = model['cost']

            demand = a + b * price
            demand = max(0.0, demand)

            product_profit = (price - cost) * demand
            total_profit += product_profit
            
        return float(total_profit)

In [13]:
historical_prices = [demand_params[p]['mean_price'] for p in demand_params.keys()]
historical_profit = ArtificialBeeColony.calculate_profit(historical_prices, demand_params)
prod_names = list(demand_params.keys())

### Первичная оптимизация цен с помощью базового ИПК

In [14]:
ABC = ArtificialBeeColony(demand_params)
best_prices, best_profit = ABC.optimize()

In [15]:
profits_history = ABC.history
iterations = list(range(1, len(profits_history) + 1))

fig_convergence = go.Figure()

fig_convergence.add_trace(go.Scatter(
    x=iterations,
    y=profits_history,
    mode='lines',
    name='Прибыль (ABC)',
    line=dict(color='#007bff', width=3),
    hovertemplate='Итерация: %{x}<br>Прибыль: %{y:.2f}<extra></extra>'
))

fig_convergence.add_hline(
    y=historical_profit,
    line_dash='dash',
    line_color='red',
    annotation_text=f'Историческая прибыль ({historical_profit:,.0f})',
    annotation_position='top right'
)

fig_convergence.update_layout(
    title='Сходимость алгоритма искусственной пчелиной колонии',
    xaxis_title='Номер итерации',
    yaxis_title='Общая прибыль (y.e.)',
    template='plotly_white',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    height=400
)

fig_convergence.show()

In [17]:
rows = []
product_names = list(demand_params.keys())

for i, name in enumerate(product_names):
    m = demand_params[name]
    p_hist = m['mean_price']
    p_opt = best_prices[i]
    q_hist = max(0.0, m['a'] + m['b'] * p_hist)
    q_opt = max(0.0, m['a'] + m['b'] * p_opt)
    profit_hist = (p_hist - m['cost']) * q_hist
    profit_opt = (p_opt - m['cost']) * q_opt

    rows.append({
        'Товар': name,
        'Цена ист.': p_hist,
        'Цена ABC': p_opt,
        'Δ цены, %': (p_opt - p_hist) / p_hist * 100,
        'Прибыль ист.': profit_hist,
        'Прибыль ABC': profit_opt,
        'Δ прибыли, %': (profit_opt - profit_hist) / profit_hist * 100 if profit_hist != 0 else np.nan,
        'Источник': m['source'],
    })

price_table = pd.DataFrame(rows).sort_values('Δ прибыли, %', ascending=False)

total_hist = price_table['Прибыль ист.'].sum()
total_abc = price_table['Прибыль ABC'].sum()
print(f"\nИтого прибыль ист.: {total_hist:.2f}")
print(f"Итого прибыль ABC: {total_abc:.2f}")
print(f"Прирост:           {(total_abc - total_hist) / total_hist * 100:+.2f}%")

price_table


Итого прибыль ист.: 17954.00
Итого прибыль ABC: 17298.38
Прирост:           -3.65%


,Товар,Цена ист.,Цена ABC,"Δ цены, %",Прибыль ист.,Прибыль ABC,"Δ прибыли, %",Источник
36,Northwoods Cranberry Sauce,38.769231,35.864522,-7.492304,443.758580,584.447203,31.703865,elasticity (-2.0)
24,Ipoh Coffee,43.042857,45.221514,5.061598,356.640816,361.098150,1.249810,elasticity (-2.0)
38,Original Frankfurter Grьne SoЯe,12.110526,12.724816,5.072361,100.836066,102.096253,1.249738,elasticity (-2.0)
44,Queso Manchego La Pastora,36.914286,38.704396,4.849369,362.814694,367.345761,1.248866,elasticity (-2.0)
49,Rцd Kaviar,14.357143,15.099889,5.173353,120.189796,121.690362,1.248497,elasticity (-2.0)
...,...,...,...,...,...,...,...,...
58,Steeleye Stout,17.000000,19.464479,14.496934,166.788889,111.564139,-33.110569,elasticity (-2.0)
34,Mozzarella di Giovanni,32.036842,23.437973,-26.840565,271.807313,137.422846,-49.441078,elasticity (-2.0)
13,Geitost,2.328125,1.618264,-30.490666,21.971680,9.075412,-58.694958,elasticity (-2.0)
43,Queso Cabrales,19.600000,13.266895,-32.311762,145.658947,46.088850,-68.358380,elasticity (-2.0)


In [18]:
price_history_array = np.array(ABC.best_prices_history)

most_expensive = price_table.loc[price_table['Цена ист.'].idxmax(), 'Товар']
cheapest = price_table.loc[price_table['Цена ист.'].idxmin(), 'Товар']

top_growth = price_table.loc[price_table['Δ прибыли, %'].idxmax(), 'Товар']
top_drop = price_table.loc[price_table['Δ прибыли, %'].idxmin(), 'Товар']

already = {most_expensive, cheapest, top_growth, top_drop}
candidates = price_table[~price_table['Товар'].isin(already)].copy()
candidates['abs_dprice'] = candidates['Δ цены, %'].abs()
top_movers = candidates.nlargest(2, 'abs_dprice')['Товар'].tolist()

show_products = list(dict.fromkeys(
    [most_expensive, cheapest, top_growth, top_drop] + top_movers
))

show_products = [p for p in show_products if p in product_names]
print(f"Автоматически выбрано {len(show_products)} товаров:")
for p in show_products:
    print(f"  {p}")

fig_prices = go.Figure()
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7', '#DDA0DD']

for idx, name in enumerate(show_products):
    i = product_names.index(name)
    fig_prices.add_trace(go.Scatter(
        x=list(range(1, len(price_history_array) + 1)),
        y=price_history_array[:, i],
        mode='lines', name=name,
        line=dict(width=2, color=colors[idx % len(colors)]),
    ))

fig_prices.update_layout(
    title=f'Эволюция цен для {len(show_products)} репрезентативных товаров',
    xaxis_title='Номер итерации', yaxis_title='Цена товара (y.e.)',
    yaxis_type='log',
    template='plotly_white', legend=dict(title='Товары'), height=450,
)
fig_prices.show()

Автоматически выбрано 6 товаров:
  Cфte de Blaye
  Geitost
  Northwoods Cranberry Sauce
  Filo Mix
  Queso Cabrales
  Tofu


In [19]:
show_df = pd.concat([
    price_table.nlargest(10, 'Δ прибыли, %'),
    price_table.nsmallest(5, 'Δ прибыли, %'),
]).drop_duplicates(subset='Товар')

x_labels = show_df['Товар'].tolist()
hist_vals = show_df['Цена ист.'].tolist()
opt_vals = show_df['Цена ABC'].tolist()

fig_comparison = go.Figure(data=[
    go.Bar(name='Историческая цена', x=x_labels, y=hist_vals,
           marker_color='lightgray',
           text=[f'{p:.2f}' for p in hist_vals], textposition='auto'),
    go.Bar(name='Оптимальная цена (ABC)', x=x_labels, y=opt_vals,
           marker_color='#007bff',
           text=[f'{p:.2f}' for p in opt_vals], textposition='auto'),
])

fig_comparison.update_layout(
    barmode='group',
    title='Топ-10 по приросту прибыли и 5 худших: исторические vs оптимальные цены',
    xaxis_title='Товар', yaxis_title='Цена (у.е.)',
    template='plotly_white', xaxis_tickangle=-45, height=500,
)
fig_comparison.show()

In [20]:
fig_scatter = go.Figure()

fig_scatter.add_trace(go.Scatter(
    x=historical_prices, y=best_prices,
    mode='markers',
    marker=dict(size=8, color='#007bff', opacity=0.7),
    text=product_names,
    hovertemplate='%{text}<br>ист.: %{x:.2f}<br>ABC: %{y:.2f}<extra></extra>',
    name='Товары',
))

min_val = min(min(historical_prices), min(best_prices))
max_val = max(max(historical_prices), max(best_prices))
fig_scatter.add_trace(go.Scatter(
    x=[min_val, max_val], y=[min_val, max_val],
    mode='lines', line=dict(dash='dash', color='gray'),
    name='y = x',
))

fig_scatter.update_layout(
    title='Историческая vs оптимальная цена (все товары)',
    xaxis_title='Историческая цена', yaxis_title='Оптимальная цена (ABC)',
    xaxis_type='log', yaxis_type='log',
    template='plotly_white', height=500,
)
fig_scatter.show()

In [21]:
num_evaluations_abc = ABC.n_evaluations

start_time_rs = time.time()

best_random_profit = -np.inf
best_random_prices = None

for _ in range(num_evaluations_abc):
    random_prices = np.random.uniform(ABC.lower_bounds, ABC.upper_bounds)
    
    current_profit = ArtificialBeeColony.calculate_profit(random_prices, demand_params)
    
    if current_profit > best_random_profit:
        best_random_profit = current_profit
        best_random_prices = random_prices

end_time_rs = time.time()
time_elapsed_rs = end_time_rs - start_time_rs

In [22]:
improvement_pct = ((best_profit - best_random_profit) / abs(best_random_profit)) * 100
print('='*50)
print("РЕЗУЛЬТАТЫ СРАВНЕНИЯ:")
print(f'Историческая прибыль:  {historical_profit:.2f}')
print(f"Прибыль ABC:           {best_profit:.2f}")
print(f"Прибыль Random Search: {best_random_profit:.2f}")
print(f"Преимущество ABC:      {improvement_pct:+.2f}%")

РЕЗУЛЬТАТЫ СРАВНЕНИЯ:
Историческая прибыль:  17954.00
Прибыль ABC:           17298.38
Прибыль Random Search: 16032.03
Преимущество ABC:      +7.90%


In [23]:
analytic_prices = []
for i, name in enumerate(product_names):
    m = demand_params[name]
    grid = np.linspace(ABC.lower_bounds[i], ABC.upper_bounds[i], 5000)
    profits = np.array([(p - m['cost']) * max(0.0, m['a'] + m['b'] * p) for p in grid])
    analytic_prices.append(grid[np.argmax(profits)])

analytic_profit = ArtificialBeeColony.calculate_profit(
    analytic_prices, demand_params
)
print(f"Аналитика (с обрезкой): {analytic_profit:.2f}")
print(f"ABC:                    {ABC.best_profit:.2f}")
print(f"Отклонение:             {(ABC.best_profit - analytic_profit) / analytic_profit * 100:+.3f}%")

Аналитика (с обрезкой): 18352.90
ABC:                    17298.38
Отклонение:             -5.746%


### Подбор оптимальных параметров для алгоритма

In [24]:
results = []

for num_bees, limit, max_iter in itertools.product(
        [20, 50, 100, 200, 500], [5, 10, 20, 50], [50, 100, 200, 500, 1000]):
    start_time = time.time()
    profits = []
    evaluations = []
    for seed in range(10):
        np.random.seed(seed)
        abc = ArtificialBeeColony(demand_params, num_bees=num_bees,
                                   max_iter=max_iter, limit=limit)
        _, p = abc.optimize()
        profits.append(p)
        evaluations.append(abc.n_evaluations)

    mean_p = float(np.mean(profits))
    results.append({
        'num_bees': num_bees,
        'limit': limit,
        'max_iter': max_iter,
        'n_evaluations': int(np.mean(evaluations)),
        'mean': mean_p,
        'std': float(np.std(profits)),
        'min': float(np.min(profits)),
        'max': float(np.max(profits)),
        'gap_to_optimum_pct': (mean_p - analytic_profit) / analytic_profit * 100,
    })
    end_time = time.time()
    time_elapsed = end_time - start_time
    print(f'num_bees={num_bees}, limit={limit}, max_iter={max_iter}: Время выполнения: {time_elapsed}, Прибыль: {mean_p}')

results_df = pd.DataFrame(results)
results_df.to_csv('sensitivity_results.csv', index=False)

print("Собрано комбинаций:", len(results_df))
print(results_df.head(12).to_string(index=False, float_format=lambda x: f"{x:.3f}"))

num_bees=20, limit=5, max_iter=50: Время выполнения: 1.0185036659240723, Прибыль: 16398.38688119339
num_bees=20, limit=5, max_iter=100: Время выполнения: 1.613386631011963, Прибыль: 16820.16096225818
num_bees=20, limit=5, max_iter=200: Время выполнения: 3.0744056701660156, Прибыль: 17080.586077576856
num_bees=20, limit=5, max_iter=500: Время выполнения: 7.647172451019287, Прибыль: 17343.53014664436
num_bees=20, limit=5, max_iter=1000: Время выполнения: 15.17839789390564, Прибыль: 17505.431793343145
num_bees=20, limit=10, max_iter=50: Время выполнения: 0.7613413333892822, Прибыль: 16523.52205939164
num_bees=20, limit=10, max_iter=100: Время выполнения: 1.495469093322754, Прибыль: 17205.992685695895
num_bees=20, limit=10, max_iter=200: Время выполнения: 3.0551576614379883, Прибыль: 17931.821754085828
num_bees=20, limit=10, max_iter=500: Время выполнения: 7.901876211166382, Прибыль: 18239.42379685492
num_bees=20, limit=10, max_iter=1000: Время выполнения: 15.546866416931152, Прибыль: 1826

In [25]:
top10 = results_df.sort_values('mean', ascending=False).head(10)
top10[['num_bees', 'limit', 'max_iter', 'mean', 'n_evaluations']].reset_index(drop=True)

,num_bees,limit,max_iter,mean,n_evaluations
0,500,50,1000,18352.897709,1000500
1,500,20,1000,18352.897578,1000538
2,200,50,1000,18352.897561,400200
3,100,50,1000,18352.897029,200100
4,200,20,1000,18352.897028,400215
5,100,20,1000,18352.896813,200108
6,50,50,1000,18352.896483,100050
7,50,20,1000,18352.896242,100054
8,20,20,1000,18352.894919,40021
9,20,50,1000,18352.894809,40020


In [26]:
fig_iter = go.Figure()

for nb in [20, 50, 100, 200, 500]:
    subset = results_df[
        (results_df['num_bees'] == nb) & (results_df['limit'] == 50)
    ].sort_values('max_iter')
    fig_iter.add_trace(go.Scatter(
        x=subset['max_iter'],
        y=subset['gap_to_optimum_pct'],
        mode='lines+markers',
        name=f'num_bees={nb}',
        hovertemplate='max_iter=%{x}<br>отклонение=%{y:.3f}%<extra></extra>',
    ))

fig_iter.add_hline(y=0, line_dash='dash', line_color='green',
                   annotation_text='Оптимум', annotation_position='bottom right')
fig_iter.update_layout(
    title='Зависимость отклонения от оптимума от max_iter (limit=50)',
    xaxis_title='max_iter',
    yaxis_title='Отклонение от оптимума, %',
    yaxis=dict(autorange='reversed'),
    template='plotly_white',
    height=450,
)
fig_iter.show()

In [35]:
fig_limit = go.Figure()

for lim in [5, 10, 20, 50]:
    subset = results_df[
        (results_df['num_bees'] == 50) & (results_df['limit'] == lim)
    ].sort_values('max_iter')
    fig_limit.add_trace(go.Scatter(
        x=subset['max_iter'],
        y=subset['gap_to_optimum_pct'],
        mode='lines+markers',
        name=f'limit={lim}',
    ))

fig_limit.add_hline(y=0, line_dash='dash', line_color='green')
fig_limit.update_layout(
    title='Зависимость отклонения от оптимума от max_iter (num_bees=50)',
    xaxis_title='max_iter',
    yaxis_title='Отклонение от оптимума, %',
    yaxis=dict(autorange='reversed'),
    template='plotly_white',
    height=450,
)
fig_limit.show()

In [28]:
fig_cost = go.Figure()

for nb in [20, 50, 100, 200, 500]:
    subset = results_df[
        (results_df['num_bees'] == nb) & (results_df['limit'] == 50)
    ].sort_values('n_evaluations')
    fig_cost.add_trace(go.Scatter(
        x=subset['n_evaluations'],
        y=subset['mean'],
        mode='lines+markers',
        name=f'num_bees={nb}',
        hovertemplate='оценок=%{x}<br>прибыль=%{y:.2f}<extra></extra>',
    ))

fig_cost.add_hline(y=analytic_profit, line_dash='dash', line_color='green',
                   annotation_text='Оптимум', annotation_position='right')
fig_cost.update_layout(
    title='Прибыль vs число оценок (limit=50)',
    xaxis_title='n_evaluations (бюджет в вызовах функции)',
    yaxis_title='Прибыль',
    xaxis_type='log',
    template='plotly_white',
    height=450,
)
fig_cost.show()

In [29]:
best_params = top10.iloc[0, :]
best_num_bees = int(best_params['num_bees'])
best_limit = int(best_params['limit'])
best_max_iter = int(best_params['max_iter'])

ABC_opti = ArtificialBeeColony(demand_params, num_bees=best_num_bees, limit=best_limit, max_iter=best_max_iter)
best_prices_opti, best_profit_opti = ABC_opti.optimize()

In [30]:
profits_history = ABC_opti.history
iterations = list(range(1, len(profits_history) + 1))

fig_convergence_opti = go.Figure()

fig_convergence_opti.add_trace(go.Scatter(
    x=iterations,
    y=profits_history,
    mode='lines',
    name='Прибыль (ABC)',
    line=dict(color='#007bff', width=3),
    hovertemplate='Итерация: %{x}<br>Прибыль: %{y:.2f}<extra></extra>'
))

fig_convergence_opti.add_hline(
    y=historical_profit,
    line_dash='dash',
    line_color='red',
    annotation_text=f'Историческая прибыль ({historical_profit:,.0f})',
    annotation_position='top right'
)

fig_convergence_opti.update_layout(
    title='Сходимость алгоритма искусственной пчелиной колонии',
    xaxis_title='Номер итерации',
    yaxis_title='Общая прибыль (y.e.)',
    template='plotly_white',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    height=400
)

fig_convergence.show()

In [31]:
rows = []
for i, name in enumerate(product_names):
    m = demand_params[name]
    p_hist = m['mean_price']
    p_opt = best_prices_opti[i]
    q_hist = max(0.0, m['a'] + m['b'] * p_hist)
    q_opt = max(0.0, m['a'] + m['b'] * p_opt)
    profit_hist = (p_hist - m['cost']) * q_hist
    profit_opt = (p_opt - m['cost']) * q_opt

    rows.append({
        'Товар': name,
        'Цена ист.': p_hist,
        'Цена ABC': p_opt,
        'Δ цены, %': (p_opt - p_hist) / p_hist * 100,
        'Прибыль ист.': profit_hist,
        'Прибыль ABC': profit_opt,
        'Δ прибыли, %': (profit_opt - profit_hist) / profit_hist * 100 if profit_hist != 0 else np.nan,
        'Источник': m['source'],
    })

opti_price_table = pd.DataFrame(rows).sort_values('Δ прибыли, %', ascending=False)

opti_total_hist = opti_price_table['Прибыль ист.'].sum()
opti_total_abc = opti_price_table['Прибыль ABC'].sum()
print(f"\nИтого прибыль ист.: {opti_total_hist:.2f}")
print(f"Итого прибыль ABC: {opti_total_abc:.2f}")
print(f"Прирост:           {(opti_total_abc - opti_total_hist) / opti_total_hist * 100:+.2f}%")

opti_price_table


Итого прибыль ист.: 17954.00
Итого прибыль ABC: 18352.90
Прирост:           +2.22%


,Товар,Цена ист.,Цена ABC,"Δ цены, %",Прибыль ист.,Прибыль ABC,"Δ прибыли, %",Источник
36,Northwoods Cranberry Sauce,38.769231,33.356696,-13.960903,443.758580,622.928298,40.375494,elasticity (-2.0)
58,Steeleye Stout,17.000000,16.020785,-5.760088,166.788889,171.647472,2.913014,elasticity (-2.0)
42,Pвtй chinois,22.400000,21.152502,-5.569188,245.178182,251.768756,2.688075,elasticity (-2.0)
19,Gula Malacca,18.133333,17.292425,-4.637361,181.635556,184.808858,1.747071,elasticity (-2.0)
47,Rhцnbrдu Klosterbier,7.379348,7.748314,4.999989,74.114319,75.040748,1.250000,elasticity (-2.0)
...,...,...,...,...,...,...,...,...
3,Carnarvon Tigers,59.722222,62.728440,5.033667,476.893004,482.853896,1.249943,elasticity (-2.0)
50,Rцssle Sauerkraut,41.975758,44.092097,5.041814,325.630119,329.700211,1.249913,elasticity (-2.0)
67,Vegie-spread,40.794118,41.243590,1.101806,427.138408,427.446359,0.072096,elasticity (-2.0)
13,Geitost,2.328125,2.343949,0.679694,21.971680,21.977696,0.027381,elasticity (-2.0)


In [32]:
opti_price_history_array = np.array(ABC_opti.best_prices_history)

most_expensive = opti_price_table.loc[opti_price_table['Цена ист.'].idxmax(), 'Товар']
cheapest = opti_price_table.loc[opti_price_table['Цена ист.'].idxmin(), 'Товар']

top_growth = opti_price_table.loc[opti_price_table['Δ прибыли, %'].idxmax(), 'Товар']
top_drop = opti_price_table.loc[opti_price_table['Δ прибыли, %'].idxmin(), 'Товар']

already = {most_expensive, cheapest, top_growth, top_drop}
opti_candidates = opti_price_table[~opti_price_table['Товар'].isin(already)].copy()
opti_candidates['abs_dprice'] = opti_candidates['Δ цены, %'].abs()
opti_top_movers = opti_candidates.nlargest(2, 'abs_dprice')['Товар'].tolist()

opti_show_products = list(dict.fromkeys(
    [most_expensive, cheapest, top_growth, top_drop] + top_movers
))

opti_show_products = [p for p in opti_show_products if p in product_names]
print(f"Автоматически выбрано {len(opti_show_products)} товаров:")
for p in show_products:
    print(f"  {p}")

fig_prices_opti = go.Figure()
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7', '#DDA0DD']

for idx, name in enumerate(show_products):
    i = product_names.index(name)
    fig_prices_opti.add_trace(go.Scatter(
        x=list(range(1, len(opti_price_history_array) + 1)),
        y=opti_price_history_array[:, i],
        mode='lines', name=name,
        line=dict(width=2, color=colors[idx % len(colors)]),
    ))

fig_prices_opti.update_layout(
    title=f'Эволюция цен для {len(show_products)} репрезентативных товаров',
    xaxis_title='Номер итерации', yaxis_title='Цена товара (y.e.)',
    yaxis_type='log',
    template='plotly_white', legend=dict(title='Товары'), height=450,
)
fig_prices_opti.show()

Автоматически выбрано 6 товаров:
  Cфte de Blaye
  Geitost
  Northwoods Cranberry Sauce
  Filo Mix
  Queso Cabrales
  Tofu


In [33]:
fig_scatter_opti = go.Figure()

fig_scatter_opti.add_trace(go.Scatter(
    x=historical_prices, y=best_prices_opti,
    mode='markers',
    marker=dict(size=8, color='#007bff', opacity=0.7),
    text=product_names,
    hovertemplate='%{text}<br>ист.: %{x:.2f}<br>ABC: %{y:.2f}<extra></extra>',
    name='Товары',
))

min_val = min(min(historical_prices), min(best_prices))
max_val = max(max(historical_prices), max(best_prices))
fig_scatter_opti.add_trace(go.Scatter(
    x=[min_val, max_val], y=[min_val, max_val],
    mode='lines', line=dict(dash='dash', color='gray'),
    name='y = x',
))

fig_scatter_opti.update_layout(
    title='Историческая vs оптимальная цена (все товары)',
    xaxis_title='Историческая цена', yaxis_title='Оптимальная цена (ABC)',
    xaxis_type='log', yaxis_type='log',
    template='plotly_white', height=500,
)
fig_scatter_opti.show()

In [34]:
print(f'Историческая прибыль: {historical_profit:.2f}')
print(f'Прибыль базового ИПК: {best_profit:.2f}')
print(f'Прибыль оптимизированного ИПК: {best_profit_opti:.2f}')

Историческая прибыль: 17954.00
Прибыль базового ИПК: 17298.38
Прибыль оптимизированного ИПК: 18352.90
